In [ ]:
!nvidia-smi

In [ ]:
!unzip "/content/Bone.zip"

In [ ]:
image_size = [224, 224]
size = [224, 224] + [3]
train_path = r'/content/Training'
valid_path = r"/content/Testing"

In [ ]:
import os

print(os.listdir(train_path))
len(train_path)

In [ ]:

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
import numpy as np
from glob import glob
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Lambda, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model


In [ ]:
efficientnet = EfficientNetB3(input_shape=image_size + [3], weights='imagenet', include_top=False)

In [ ]:
for layer in efficientnet.layers:
    print(layer.trainable)
for layer in efficientnet.layers:
    layer.trainable = False
for layer in efficientnet.layers:
    layer.name, layer.trainable

In [ ]:
efficientnet.summary()
folder = glob(r'/content/Training')

In [ ]:
import os

path = r'/content/Training'
num_of_class = len(os.listdir(path))
print(num_of_class)

In [ ]:
model = Sequential()

model.add(EfficientNetB3)
model.add(GlobalAveragePooling2D())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_of_class, activation='softmax'))

In [ ]:
model.summary()

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [ ]:

train_datagen = ImageDataGenerator(
    # CHANGE THIS LINE: Use efficientnet preprocessing instead of vgg16
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    shear_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    rotation_range=20
)
test_datagen = ImageDataGenerator(
    # CHANGE THIS LINE as well
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

In [ ]:
training_set = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)
test_set = test_datagen.flow_from_directory(
    valid_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
r = model.fit(
    training_set,
    validation_data=test_set,
    epochs=10,
    steps_per_epoch=len(training_set),
    validation_steps=len(test_set)

)

In [ ]:

model.evaluate(test_set)
from tensorflow.keras.models import load_model

# Save in modern format
model.save('model_efficientnet_b3.keras')

# Load it back
model = load_model('model_efficientnet_b3.keras')
from tensorflow.keras.preprocessing import image

img = r'/content/Training/glioma/Tr-gl_100.jpg'
img = image.load_img(img, target_size=(224, 224))
img
x = image.img_to_array(img)
x
x.shape
x = x / 255
from keras.applications.efficientnet import preprocess_input

img_data = preprocess_input(x)
img_data = np.expand_dims(img_data, axis=0)
output = model.predict(img_data)
output
import numpy as np

result = model.predict(img_data)
pred_index = np.argmax(result)

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
prediction = class_names[pred_index]

if prediction == 'notumor':
    final_result = "No Tumor"
else:
    final_result = "Tumor"

print("Class:", prediction)
print("Final:", final_result)